# Week6_ex4 - Force Between Two Magnets

Two N42 magnets facing each other across a gap, magnetized in opposite directions so they attract -- sweeps the gap distance with Magnetostatic and logs force/torque on the lower magnet at each step to build a force-vs-distance table.

Note: the original notebook computed a single fixed-gap point and its report already referenced a swept "X" position variable that was never actually defined anywhere. This version registers the gap as a real design variable and adds the missing parametric sweep so the report matches what it's asking for.

In [ ]:
import pandas as pd
import os
import ansys.aedt.core
import math

In [ ]:
## Variables ##

Gap = 5      # gap between the two magnets -- swept below, this is just the starting value
Width = 20   # magnet width
Height = 10  # magnet height
Depth = 100  # magnet depth

In [ ]:
# 1. Launch AEDT
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week6_ex4"   # matches this exercise's folder name

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week6_ex4"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Model units ##

M3D.modeler.model_units = "mm"

In [ ]:
## Design variable definitions ##

# Gap is registered as a real design variable so it can be swept below --
# Width/Height/Depth are registered too, mainly so the geometry expressions
# can reference them by name instead of baking in raw numbers.
M3D["Gap"] = f"{Gap}mm"
M3D["Width"] = f"{Width}mm"
M3D["Height"] = f"{Height}mm"
M3D["Depth"] = f"{Depth}mm"

In [ ]:
## Draw the magnets ##

# add the N42 magnet material if it isn't already in the library
if not M3D.materials.exists_material("N42"):
   M3D.materials.add_material("N42")
N42 = M3D.materials["N42"]
N42.permeability.value = 1.05
N42.set_magnetic_coercivity("-970116", x=0, y=1, z=0)  # default direction, adjusted per magnet below

# magnets -- lower magnet's position uses the Gap design variable so the
# parametric sweep below can actually move it
origin = [0, "-(Height+Gap)", 0]
sizes = ["Width", "Height", "Depth"]
magnet1 = M3D.modeler.create_box(origin=origin, sizes=sizes, name="magnet1", material="N42")

origin = [0, 0, 0]
sizes = ["Width", "Height", "Depth"]
magnet2 = M3D.modeler.create_box(origin=origin, sizes=sizes, name="magnet2", material="N42")

In [ ]:
## Magnetize the magnets ##

## lower magnet (magnet1) ##
# local coordinate system attached to the magnet's face
origin = magnet1.faces[0].center
M3D.modeler.create_coordinate_system(origin=origin, reference_cs='Global', name="FaceCS1", 
                                     mode='axis', view='iso', x_pointing=None, y_pointing=None, psi=0, theta=0, phi=0, u=None)

magnet1.part_coordinate_system = "FaceCS1"          # attach the magnet to its face coordinate system
M3D.modeler.set_working_coordinate_system("Global") # back to the global coordinate system

# check and set the magnetization direction
N42 = M3D.materials.exists_material(material="N42")
N42.set_magnetic_coercivity('-970116', x=0, y=1, z=0)   # (+Y)

## upper magnet (magnet2) ##
# local coordinate system attached to the magnet's face
origin = magnet2.faces[0].center
M3D.modeler.create_coordinate_system(origin=origin, reference_cs='Global', name="FaceCS2", 
                                     mode='axis', view='iso', x_pointing=None, y_pointing=None, psi=0, theta=0, phi=0, u=None)

magnet2.part_coordinate_system = "FaceCS2"
M3D.modeler.set_working_coordinate_system("Global") # back to the global coordinate system

# check and set the magnetization direction
N42.set_magnetic_coercivity('-970116', x=0, y=1, z=0)   # (+Y)

In [ ]:
M3D.assign_force(assignment=magnet1, coordinate_system='Global', is_virtual=True, force_name="Force1")

# torque about X
M3D.assign_torque(assignment=magnet1, coordinate_system="Global", axis="X", is_virtual=True, torque_name="Torque_X")

# torque about Y
M3D.assign_torque(assignment=magnet1, coordinate_system="Global", axis="Y", is_virtual=True, torque_name="Torque_Y")

# torque about Z
M3D.assign_torque(assignment=magnet1, coordinate_system="Global", axis="Z", is_virtual=True, torque_name="Torque_Z")

In [ ]:
## Region ##

region = M3D.modeler.create_region(pad_value=500, pad_type='Percentage Offset', name='Region')

In [ ]:
## Analyze Setup ##

# setup object
my_setup = M3D.create_setup(name="Setup1")

# check the analysis setup properties for the current solution type --
# they're stored as a dictionary on the setup object's props attribute
display(my_setup.props)

In [ ]:
# adjust whichever of the props displayed above is needed
# MaximumPasses: controls solve accuracy through repeated mesh refinement passes
my_setup.props['MaximumPasses'] = 25

In [ ]:
## Sweep over Gap (Python loop, not Optimetrics) ##

# NOTE: switched from M3D.parametrics.add(...) to a plain Python loop. The
# Optimetrics sweep completed fine ("Parametric Analysis is done"), but neither
# the GUI Data Table report nor get_solution_data() could pull Force1.Force_mag/
# etc. across the swept variations ("No trace data for quantities...") -- Maxwell
# apparently needs a separate "Expression Cache" set up per-quantity for Optimetrics
# sweeps to retain scalar output-variable data at each point, which isn't something
# assign_force()/assign_torque() set up automatically. Re-solving one Gap value at a
# time and reading the result immediately after each solve sidesteps this entirely,
# matching the pattern already used elsewhere in this project (e.g. Week5_ex3).

gap_values = list(range(2, 21, 2))  # 2mm to 20mm, 2mm step -- same range as before

rows = []
for g in gap_values:
    M3D["Gap"] = f"{g}mm"
    my_setup.analyze()

    data = M3D.post.get_solution_data(
        expressions=["Force1.Force_mag", "Force1.Force_x", "Force1.Force_y", "Force1.Force_z"],
        setup_sweep_name="Setup1 : LastAdaptive",
    )

    rows.append({
        "Gap_mm": g,
        "Force_mag_N": data.data_real("Force1.Force_mag")[0],
        "Force_x_N": data.data_real("Force1.Force_x")[0],
        "Force_y_N": data.data_real("Force1.Force_y")[0],
        "Force_z_N": data.data_real("Force1.Force_z")[0],
    })
    print(rows[-1])

df = pd.DataFrame(rows)

csv_name = "Week6_ex4_force.csv"
csv_path = csv_name  # saved flat, next to the notebook -- same convention as Images/
df.to_csv(csv_path, index=False)

df.head()

In [ ]:
## Save project ##

M3D.save_project()